In [8]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import os
import timeit
plt.ion()   # interactive mode

In [9]:
def train(model, device, train_loader, optimizer, criterion, scheduler):    
    EPOCHS = 50
    for epoch in range(EPOCHS):
        losses = []
        running_loss = 0
        for i, inp in enumerate(train_loader):
            inputs, labels = inp
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            losses.append(loss.item())
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
            if i%100 == 0 and i > 0:
                print(f'Loss [{epoch+1}, {i}](epoch, minibatch): ', running_loss / 100)
                running_loss = 0.0

        avg_loss = sum(losses)/len(losses)
        scheduler.step(avg_loss)

    print('Training Done')

In [10]:
def test(model, device, test_loader):
    correct = 0
    total = 0

    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('Accuracy on 10,000 test images: ', 100*(correct/total), '%')

In [16]:
import os
import ssl
from pathlib import Path
from urllib.request import urlopen

candidates = [
    "/etc/pki/tls/certs/ca-bundle.crt",
    "/etc/ssl/certs/ca-certificates.crt",
    "/etc/ssl/ca-bundle.pem",
]

bundle = next((p for p in candidates if Path(p).is_file()), None)

if bundle is None:
    print("No system certificate bundle found. Ask CARC support for its path.")
else:
    os.environ["SSL_CERT_FILE"] = bundle
    context = ssl.create_default_context(cafile=bundle)
    url = "https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz"
    with urlopen(url, context=context, timeout=30) as response:
        print("Connection successful:", response.status)

Connection successful: 200


In [12]:
import urllib.request
from torchvision import datasets

# Reuse the SSL context from your successful connection test.
opener = urllib.request.build_opener(
    urllib.request.HTTPSHandler(context=context)
)
urllib.request.install_opener(opener)

dataset1 = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

dataset2 = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

print(f"Loaded {len(dataset1)} training and {len(dataset2)} test images.")

NameError: name 'transform' is not defined

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
dataset_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=64, num_workers=4, pin_memory=True)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=64, num_workers=4, pin_memory=True)

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

55.7%

In [15]:
import urllib.request
from torchvision import datasets

# Reuse the SSL context from your successful connection test.
opener = urllib.request.build_opener(
    urllib.request.HTTPSHandler(context=context)
)
urllib.request.install_opener(opener)

dataset1 = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

dataset2 = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

print(f"Loaded {len(dataset1)} training and {len(dataset2)} test images.")

NameError: name 'transform' is not defined

In [ ]:
def imshow(inp, title=None):
    """Display image for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.5, 0.5, 0.5])
    std = np.array([0.5, 0.5, 0.5])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated

# Get a batch of training data
inputs, classes = next(iter(train_loader))

# Make a grid from batch
out = torchvision.utils.make_grid(inputs)
class_names = dataset_train.classes
imshow(out)

Fine-tuning 

In [ ]:
model= models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model.fc.in_features

# Here the size of each output sample is set to 10.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.
model.fc = nn.Linear(num_ftrs, 10)
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor = 0.1, patience=5)

In [ ]:
start_time = timeit.default_timer()
train(model, device, train_loader, optimizer, criterion, scheduler)
print(timeit.default_timer() - start_time)
test(model, device, test_loader)